In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Standard
import random

# Third Party
from datasets import load_dataset
from openai import OpenAI
from rich import print
from rich.panel import Panel
from sklearn.metrics import classification_report

# First Party
from sdg_hub import Flow

/home/lab/esivaram/sdg_hub/venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Annotation with AG news dataset


In this tutorial, you’ll learn how to create your own custom data generation flow using SDG Hub. More specifically, with this flow, we will use a language model to **annotate user-generated text** with topic labels — specifically using the [AG News dataset](https://huggingface.co/datasets/fancyzhx/ag_news) from Hugging Face.

We’ll go step by step through a progressively improving flow. Each stage builds on the previous one, giving you a practical sense of how synthetic labeling can evolve from simple heuristics to highly customized and reliable data generation.

### 🔍 Understand the Task
Before we write any prompts or code, we’ll take time to understand what we want the model to learn. For this exercise, the task is **topic annotation** — assigning one of ten possible categories (e.g., "Science & Mathematics", "Sports", "Politics & Government") to a user-submitted question or paragraph.

### 🛠️ Build a Basic Annotation Flow
We’ll start by creating a minimal flow that takes a small number of seed examples and uses them to generate topic labels on the unlabeled data. This will use default prompts and simple scoring logic to simulate how annotation works.

### 🎯 Improve with Better Examples
Next, we’ll refine the flow by enhancing the **seed examples** and the **prompt**. Better examples = better generations. You’ll see how even a small change in phrasing, structure, or label clarity can dramatically improve output quality.

### ✏️ Customize with Your Own Prompts
Finally, we’ll show you how to take full control by writing your own prompts from scratch. This allows you to inject task-specific instructions, formatting rules, or even domain tone — enabling the model to generalize better and reduce noise in the generated labels.

Let’s get started by loading a sample of the dataset and identifying what task we want the model to learn.

In [3]:
dataset = load_dataset("fancyzhx/ag_news")

train_data = dataset["train"].shuffle(seed=42).select(range(500))
test_data = dataset["test"].shuffle(seed=42).select(range(100))

# map the labels to the category names
label_map = train_data.features['label'].names

train_data = train_data.map(lambda x: {"category": label_map[x["label"]]})
test_data = test_data.map(lambda x: {"category": label_map[x["label"]]})

In [4]:
# Group examples by category
examples_by_category = {}
for item in train_data:
    category = item['category']
    if category not in examples_by_category:
        examples_by_category[category] = []
    examples_by_category[category].append(item['text'])

# Print one example from each category in a panel
for category, examples in examples_by_category.items():
    print(Panel(examples[0], title=f"Category: {category}", expand=False))


╭──────────────────────────────────────────────── Category: World ────────────────────────────────────────────────╮
│ Bangladesh paralysed by strikes Opposition activists have brought many towns and cities in Bangladesh to a      │
│ halt, the day after 18 people died in explosions at a political rally.                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Category: Sports ────────────────────────────────────────────────╮
│ Desiring Stability Redskins coach Joe Gibbs expects few major personnel changes in the offseason and wants to   │
│ instill a culture of stability in Washington.                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Category: Sci/Tech ───────────────────────────────────────────────╮
│ U2 pitches for Apple New iTunes ads airing during baseball games Tuesday will feature the advertising-shy Irish │
│ rockers.                                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Category: Business ───────────────────────────────────────────────╮
│ Economy builds steam in KC Fed district The economy continued to strengthen in September and early October in   │
│ the Great Plains and Rocky Mountain regions covered by the Tenth Federal Reserve District, the Federal Reserve  │
│ Bank of Kansas City said Wednesday.                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Simple Data Annotation Pipeline

In this section, we’ll create our **first working pipeline** to perform annotation using a language model. The goal is to simulate how the model can annotate raw user queries with topic labels using a minimal configuration.

### Recap: How  SDG_HUB Works

```mermaid
flowchart LR
    A[Flow] --> B[Blocks] --> C[Prompts]
    C --> D[Synthetic Data!]
```

### Flow

Below is a minimal flow that uses a single LLMBlock to annotate raw questions. We’re using guided decoding with a fixed label set to keep model outputs controlled and consistent. Pay attention to the generation params such as temperature, max_tokens and guided_choice under LLMChatBlock. These parameters are important to influence the generation outcomes in a consistent (0 temperature) and constrained (guided choices) way.

```yaml
metadata:
  name: Simple Annotation
  description: A simple annotation flow for categorizing text
blocks:
- block_type: PromptBuilderBlock
  block_config:
    block_name: simple_annotation_prompt
    input_cols:
    - text
    output_cols:
    - annotation_prompt
    prompt_config_path: ../../../../examples/annotation/flows/simple_annotation/prompt.yaml
    format_as_messages: true
- block_type: LLMChatBlock
  block_config:
    block_name: simple_annotation
    input_cols:
    - annotation_prompt
    output_cols:
    - raw_output
    temperature: 0.0
    max_tokens: 5
    extra_body:
      guided_choice:
        - World
        - Sports
        - Business
        - Sci/Tech
- block_type: TextParserBlock
  block_config:
    block_name: parse_annotation_output
    input_cols:
    - raw_output
    output_cols:
    - output
    start_tags:
    - ''
    end_tags:
    - '' 

```

### Prompt

This prompt teaches the model to take in a freeform query and return a single topic label. Since we’re using guided decoding, we’re keeping the format minimal and relying on constrained sampling to enforce label consistency.

```yaml
- role: system
  content: You are a helpful assistant that annotates text.

- role: user
  content: |
    Task Description: Data Annotation

    Here is the query for annotation:
    {{text}}

```

This prompt passes the raw text to the model with minimal guidance — think of it as a baseline to test how much the model already understands the task when constrained to a limited label set.

### What This Does
* Loads a batch of input text (e.g., AG news articles)
* Passes each query into the prompt under the {{text}} template
* Uses guided decoding (via xgrammar) to force the output to be one of the specified topic labels
* Outputs predictions in the output column

This is the simplest version of an annotation pipeline — no examples, no complex prompting — just a structured flow powered by modular blocks.

Let's test it out!

In [5]:
openai_api_key = "EMPTY" # replace with your inference server api key
openai_api_base = "http://0.0.0.0:8000/v1" # replace with your inference server endpoint


client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

models = client.models.list()
teacher_model = models.data[0].id

# Test the connection with a simple completion
response = client.chat.completions.create(
    model=teacher_model,
    messages=[{"role": "user", "content": "Hello!"}],
    temperature=0.0,
    max_tokens=10
)
completion = response.choices[0].message.content

print(f"Connection successful! {teacher_model}: {completion}")

Connection successful! meta-llama/Llama-3.3-70B-Instruct: Hello. How can I help you today?

### Run the Simple Annotation Pipeline

In [6]:
# Load the flow
simple_annotation_flow = Flow.from_yaml("flows/simple_annotation/flow.yaml")

[19:07:11] INFO     Loading flow from: flows/simple_annotation/flow.yaml                                ]8;id=347072;file:///workspace/home/lab/esivaram/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=407093;file:///workspace/home/lab/esivaram/sdg_hub/src/sdg_hub/core/flow/base.py#140\140]8;;\

In [ ]:
# Set the model config here (point to your hosted inference server or any inference API accordingly)
simple_annotation_flow.set_model_config(model="hosted_vllm/meta-llama/Llama-3.3-70B-Instruct", api_base="http://localhost:8000/v1", api_key="")

In [ ]:
generated_data = simple_annotation_flow.generate(test_data)

### Evaluation

Now that we’ve generated synthetic topic labels using our annotation pipeline, it’s time to evaluate how well the model performed. The goal is to compare the predicted labels against the **true labels** from the dataset using standard classification metrics.

We’ll use `sklearn.metrics.classification_report`, which provides precision, recall, F1-score, and support for each class.


In [10]:
print(classification_report(generated_data["category"], generated_data["output"]))

/home/lab/esivaram/sdg_hub/venv/lib64/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/lab/esivaram/sdg_hub/venv/lib64/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/lab/esivaram/sdg_hub/venv/lib64/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

precision    recall  f1-score   support

    Business       0.39      1.00      0.56        32
    Sci/Tech       0.00      0.00      0.00        19
      Sports       0.92      0.44      0.60        27
       World       0.50      0.09      0.15        22

    accuracy                           0.46       100
   macro avg       0.45      0.38      0.33       100
weighted avg       0.48      0.46      0.37       100

## Improving Results with Examples and Custom Prompts

Our initial pipeline used a **zero-shot approach** — the model was given the task, a fixed label set, and some input text, but **no examples of how to perform the task**. While this baseline gives us a useful starting point, it has clear limitations:

- The model may rely on generic heuristics or surface patterns that don’t generalize well.
- It can confuse similar categories (e.g., "World" vs. "Business") without knowing how they're typically used.
- Without guidance, the model may underperform on edge cases or ambiguous queries.


### Why Examples Matter

In-context examples act as **training demonstrations** — they teach the model how to think, how to respond, and how to structure its output.

With even a few high-quality seed examples, we can:
- **Disambiguate confusing labels** by showing contrasting cases
- **Guide tone and formatting**, especially for structured tasks
- **Bias the model toward higher precision** by anchoring it to gold examples

Think of examples as the foundation for aligning the model to your task — they provide **task intent**, **style**, and **semantic anchors** for generation.


### What We’ll Do Next

We’ll now enhance our prompt by adding **4 examples** that cover a variety of labels from the dataset. These examples will be inserted into the prompt file used by the `LLMChatBlock`.

You’ll then rerun the same flow and compare the results — and see how a few carefully chosen examples can dramatically improve both **accuracy** and **label consistency**.


```yaml
- role: system
  content: You are an expert text classifier trained to label questions from online forums.

- role: user
  content: |
    Task Description: You will be given a text and you need to annotate it with one of the following categories: World, Sports, Sci/Tech and Business

    Please follow these rules when performing the classification:
    - Focus on the main topic, not peripheral mentions
    - Choose the most specific applicable category
    - Only choose category label per question

    Examples:

    Text: Bangladesh paralysed by strikes Opposition activists have brought many towns and cities in Bangladesh to a halt, the day after 18 people died in explosions at a political rally.
    Category: World

    Text: Desiring Stability Redskins coach Joe Gibbs expects few major personnel changes in the offseason and wants to instill a culture of stability in Washington.
    Category: Sports

    Text: A Cosmic Storm: When Galaxy Clusters Collide Astronomers have found what they are calling the perfect cosmic storm, a galaxy cluster pile-up so powerful its energy output is second only to the Big Bang.
    Category: Sci/Tech

    Text: Economy builds steam in KC Fed district The economy continued to strengthen in September and early October in the Great Plains and Rocky Mountain regions covered by the Tenth Federal Reserve District, the Federal Reserve Bank of Kansas City said Wednesday.
    Category: Business

    Here is the query for annotation:

    Text: {{text}}
    Category:

```

### Run the Flow with Examples and Custom Prompts

In [11]:
# Load the flow
detailed_annotation_flow = Flow.from_yaml("flows/detailed_annotation/flow.yaml")

[19:08:38] INFO     Loading flow from: flows/detailed_annotation/flow.yaml                              ]8;id=520657;file:///workspace/home/lab/esivaram/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=829320;file:///workspace/home/lab/esivaram/sdg_hub/src/sdg_hub/core/flow/base.py#140\140]8;;\

In [ ]:
detailed_annotation_flow.set_model_config(model="hosted_vllm/meta-llama/Llama-3.3-70B-Instruct", api_base="http://localhost:8000/v1", api_key="")

In [ ]:
generated_data = detailed_annotation_flow.generate(test_data)

### Evaluation

In [14]:
print(classification_report(generated_data["category"], generated_data["output"]))

precision    recall  f1-score   support

    Business       0.80      1.00      0.89        32
    Sci/Tech       0.93      0.68      0.79        19
      Sports       0.93      1.00      0.96        27
       World       0.94      0.73      0.82        22

    accuracy                           0.88       100
   macro avg       0.90      0.85      0.87       100
weighted avg       0.89      0.88      0.88       100

## ✅ Summary: What You’ve Learned

In this tutorial, you built a complete data annotation flow — starting from scratch and evolving into a robust, high-accuracy system. Along the way, you learned the skill of adapting Large Language Models for a specific use-case!

### 🚀 What’s Next?

* Extend the flow further! - Add an evaluation step
* Try it out on your own data!